# Reproduce and audit the MSR 2027 benchmark tables

This notebook recalculates saved annotation and prediction tables. It does not acquire repositories or call an LLM. Source CSVs have exactly seven columns and 101 rows. The latest named heuristic sheets are selected; superseded sheets are omitted.

**Important:** CSV binary completeness is not runtime scan completeness. The paper's heuristic cohort must be reconstructed from original saved runs, not guessed from the worksheet. All 100 repositories informed development.


In [1]:
from pathlib import Path
import importlib.util
import json

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "results/benchmark-100/reproduce_paper.py").exists())
HERE = ROOT / "results/benchmark-100"
spec = importlib.util.spec_from_file_location("paper_reproduction", HERE / "reproduce_paper.py")
analysis = importlib.util.module_from_spec(spec)
spec.loader.exec_module(analysis)

# Set these to the original local files when available. Do not rename or edit manifests.
ORIGINAL_RUN = None  # e.g. ROOT / "data/runs/benchmark-100"
CORPUS_PREDICTIONS = None  # e.g. ROOT / "results/historical-386/predictions.csv"
report = analysis.analyze(ORIGINAL_RUN, CORPUS_PREDICTIONS)
print("Workbook SHA-256:", report["inputs"]["source_workbook_sha256"])
for item in report["inputs"]["sheets"]:
    print(item["role"], "<-", item["source_sheet"], item["range"], "101 rows x 7 columns")


Workbook SHA-256: 9331524372765d6c4898cdede32b2c9d0b654d4638d18c6f46358fcfbf9928a1
reviewer1 <- research_lifecycle_benchmark_10 A1:G101 101 rows x 7 columns
reviewer2 <- Hoja 10 A1:G101 101 rows x 7 columns
agreement <- agreement A1:G101 101 rows x 7 columns
final_agreement <- final_agreement A1:G101 101 rows x 7 columns
llm_documentation <- Hoja 11 A1:G101 101 rows x 7 columns
llm_code <- Hoja 12 A1:G101 101 rows x 7 columns
llm_combined <- Hoja 13 A1:G101 101 rows x 7 columns
heuristics_documentation <- benchmark_100_heuristics_doc_la A1:G101 101 rows x 7 columns
heuristics_code <- benchmark_100_heuristics_code_l A1:G101 101 rows x 7 columns
heuristics_combined <- benchmark_100_heuristics_both_l A1:G101 101 rows x 7 columns


## Human annotations and Table 3

Independent reviewer labels are compared by normalized repository identity, never by assumed row position. The agreement CSV is checked against all 600 independent comparisons. Final consensus is a separate reference.


In [2]:
annotations = report["annotations"]
claims = json.loads((HERE / "paper_claims.json").read_text())
for row in annotations["per_stage"]:
    assert row["agreement"] == claims["table_3"][row["stage"]]
    print(f"{row['stage']:18} agreement={row['agreement']:3} disagreement={row['disagreement']:3} final positive={row['final_positive']:3}")
print("Totals:", {key: value for key, value in annotations.items() if key != "per_stage"})
assert annotations["agreement"] == 354 and annotations["disagreement"] == 246
assert annotations["final_positive_labels"] == 384 and annotations["all_six_positive_repositories"] == 12


Collection         agreement= 53 disagreement= 47 final positive= 36
Processing         agreement= 55 disagreement= 45 final positive= 44
Method             agreement= 53 disagreement= 47 final positive= 71
Experimentation    agreement= 72 disagreement= 28 final positive= 88
Evaluation         agreement= 53 disagreement= 47 final positive= 66
Dissemination      agreement= 68 disagreement= 32 final positive= 79
Totals: {'agreement': 354, 'disagreement': 246, 'final_positive_labels': 384, 'final_negative_labels': 216, 'all_six_positive_repositories': 12}


## Metrics from the selected latest sheets

A repository enters a CSV comparison only if all six cells are literal binary values. Blank and Unverified labels remain unknown. Precision/recall/F1 and exact match are calculated against final consensus. Per-stage confusion counts and baselines are saved in the output JSON. These CSV denominators must not be presented as scan-complete cohort sizes.


In [3]:
print(f"{'Source':28} {'N':>3} {'P':>8} {'R':>8} {'Micro-F1':>9} {'Macro-F1':>9} {'Exact':>8}")
for role, result in report["workbook_metrics"].items():
    m = analysis.compact(result)
    print(f"{role:28} {m['repositories']:3} {m['micro_precision']:8.5f} {m['micro_recall']:8.5f} {m['micro_f1']:9.5f} {m['macro_f1']:9.5f} {m['exact_match']:8.5f}")
    print("  excluded:", result["excluded_repository_ids"])
    print("  always-positive micro-F1:", result["baselines"]["always_positive"]["micro"]["f1"])


Source                         N        P        R  Micro-F1  Macro-F1    Exact
llm_documentation             98  0.77855  0.88594   0.82878   0.80979  0.19388
  excluded: ['emergent-behaviors-in-biology/community-simulator', 'ihincks/nv-adaptive']
  always-positive micro-F1: 0.78134715
llm_code                      63  0.66292  0.97925   0.79062   0.76694  0.15873
  excluded: ['aquemy/dpso_experiments', 'aviranlab/sleq', 'b5strbal/macaw', 'christopherwilks/snaptron-experiments', 'ciaranm/cp2015-subgraph-isomorphism', 'cristal-smac/atom', 'diglesia/delaiglesiaetal_data', 'dmlb2000/eetup', 'drbenvincent/darc-experiments-matlab', 'drgriffis/sbd-evaluation', 'eglxiang/ytf', 'emergent-behaviors-in-biology/community-simulator', 'freedisciplina/collisionoffset', 'gmarceaucaron/natural-langevin-dynamics-for-neural-networks', 'ihincks/nv-adaptive', 'irit-melodi/irit-rst-dt', 'irit-melodi/irit-stac', 'jmschrei/pypore', 'joefresna/plosone-reina2015', 'liusi2019/ocd', 'md-k-sarker/ecii-expr', 'mi

## Same-cohort comparisons

The intersection is calculated separately for the three heuristic sheets and the three LLM sheets. It is the intersection of binary-complete source vectors, not proof of a common execution snapshot or independent code accuracy.


In [4]:
for family, results in report["common_binary_complete_cohorts"].items():
    print(family)
    for role, result in results.items():
        print(role, "N=", result["repository_count"], "micro-F1=", result["micro"]["f1"], "macro-F1=", result["macro"]["f1"])


llm
llm_documentation N= 63 micro-F1= 0.84540117 macro-F1= 0.82007506
llm_code N= 63 micro-F1= 0.79061977 macro-F1= 0.76694479
llm_combined N= 63 micro-F1= 0.79338843 macro-F1= 0.76978086
heuristics
heuristics_documentation N= 98 micro-F1= 0.69426752 macro-F1= 0.61293073
heuristics_code N= 98 micro-F1= 0.41035857 macro-F1= 0.35051531
heuristics_combined N= 98 micro-F1= 0.74744526 macro-F1= 0.67422638


## Paper consistency checks

Table 4 reports 96/95/95 scan-complete repositories. The worksheet's latest exports yield 99/100/99 binary-complete vectors, so this notebook cannot recover Table 4 without original run status. Table 6 is reproducible apart from the combined micro-F1 rounding: 0.79454927 rounds to 0.795, while the draft prints 0.794. Table 7 still contains superseded heuristic values. No repository subset is selected to force agreement with the paper.


In [5]:
differences = [row for row in report["paper_comparison"] if row["status"] != "matches"]
for row in differences:
    print(row["table"], row["role"], row["metric"], "paper=", row["paper"], "computed=", row["computed"], row["status"])
print("Original-run Table 4:", "recomputed" if report["original_run_metrics"] else "unavailable: supply ORIGINAL_RUN")
print("386-repository Table 5:", "recomputed" if report["corpus_stage_counts"] else "unavailable: supply CORPUS_PREDICTIONS")
if report["original_run_paper_comparison"]:
    print(json.dumps(report["original_run_paper_comparison"], indent=2))
if report["corpus_paper_comparison"]:
    print(json.dumps(report["corpus_paper_comparison"], indent=2))


table_4 heuristics_documentation repositories paper= 96 computed= 99 differs
table_4 heuristics_documentation micro_precision paper= 0.865 computed= 0.8627451 differs
table_4 heuristics_documentation micro_recall paper= 0.591 computed= 0.58047493 differs
table_4 heuristics_documentation micro_f1 paper= 0.702 computed= 0.69400631 differs
table_4 heuristics_documentation macro_f1 paper= 0.62 computed= 0.61132781 differs
table_4 heuristics_documentation exact_match paper= 0.115 computed= 0.11111111 differs
table_4 heuristics_documentation label_accuracy paper= 0.679 computed= 0.6734006734006734 differs
table_4 heuristics_documentation coverage paper= 0.96 computed= 0.99 differs
table_4 heuristics_code repositories paper= 95 computed= 100 differs
table_4 heuristics_code micro_precision paper= 0.821 computed= 0.81395349 differs
table_4 heuristics_code micro_recall paper= 0.277 computed= 0.2734375 differs
table_4 heuristics_code micro_f1 paper= 0.414 computed= 0.40935673 differs
table_4 heur

## Save reproducible exports

Outputs include per-stage metrics, own/common-cohort metrics and baselines, the claim comparison, and a status note. Optional original-run and 386 exports are generated only when the corresponding source data are supplied. The LLM prompts and shared-conversation provenance are in ../../prompts/.


In [6]:
analysis.write_outputs(report, HERE / "analysis")
print("Saved:", "results/benchmark-100/analysis")
for limitation in report["limitations"]:
    print("-", limitation)


Saved: results/benchmark-100/analysis
- Workbook CSV binary completeness does not establish runtime scan completeness.
- Paper Table 4 requires original result.json files to reconstruct the 96/95/95 scan-complete cohorts.
- Paper Table 5 requires the original 386-repository predictions export.
- Paper Table 7 retains superseded heuristic values and is inconsistent with Table 4.
- The combined LLM micro-F1 rounds to 0.795; the paper prints 0.794.
- All 100 repositories informed heuristic development; no held-out claim.
